# Test 

In [ ]:
# Run once in this environment: pip install -e .
from pathlib import Path

import torch

import src
from src.models.CNN import CNN
from src.test.utils import (
    display_formatted_results,
    evaluate_multilabel_performance,
    run_inference,
)

PROJECT_ROOT = Path(src.__file__).resolve().parents[1]
MODEL_WEIGHTS = PROJECT_ROOT / "src" / "models" / "saved_weights" / "CNN_v1" / "best_val.pt"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


## Run on A Touch of Zen Dataset

In [ ]:
test_manifest_csv = PROJECT_ROOT / "data" / "test" / "a-touch-of-zen.csv"

preds_arr, gts_arr, sample_ids, audio_cfg, valid_labels, label_to_idx = run_inference(
    model_cls=CNN,
    model_kwargs={"in_ch": 2},
    model_weights_path=MODEL_WEIGHTS,
    device=DEVICE,
    test_manifest_csv=test_manifest_csv,
)


## Tune classification threshold probability 

Optimising macro F1 means:
- You care equally about rare and common instruments

Optimising micro F1 means:
- You care about overall instrument detection performance
- Every correctly detected instrument occurrence matters equally
- Common instruments dominate

In [ ]:
from src.test.utils import find_best_threshold

best_t = find_best_threshold(preds_arr, gts_arr, valid_labels, show_plot=False)
print(f"Best threshold found: {best_t:.2f}")
# print(valid_labels)


In [ ]:
from src.test.utils import evaluate_multilabel_performance


threshold_probability = best_t
# threshold_probability = 0.9

results = evaluate_multilabel_performance(
    all_preds=preds_arr,
    all_gt=gts_arr,
    class_list=valid_labels,
    sample_ids=sample_ids,
    threshold=threshold_probability,
    debug=True,
)

### Classification Report


In [ ]:
display_formatted_results(results)